# Grid-Connected Tidal Energy Case Study

This case study demonstrates how VITAL can be used to explore a representative grid-connected tidal energy system for HDPS.

The example walks through:

- tidal resource loading
- rotor performance loading
- baseline design setup
- site comparison
- dynamic simulation
- physical constraint checking
- LCOE estimation
- design optimization

The goal is to show how the software helps answer practical questions such as:

- Which site looks most promising?
- Is the design physically feasible?
- What drives the cost of energy?
- Can the design be improved?

This case study is intended for a broad audience and uses representative inputs to illustrate the workflow. The results should be interpreted as screening-level estimates, not final design recommendations.

The conclusions depend on the selected rotor data, vessel/platform assumptions, cost model, optimization bounds, and tidal data source.

## 1. Why this tool matters

Tidal energy development requires decisions about both where to deploy a system and how to design it.

A promising site should have:

- sufficient tidal flow,
- practical mooring and cable conditions,
- and a system configuration that remains physically feasible and economically viable.

VITAL supports early-stage screening by combining:

- site data,
- turbine performance curves,
- vessel/platform properties,
- dynamic simulation,
- constraint checks,
- and levelized cost of energy (LCOE) calculations.

In this case study, we focus on a representative grid-connected system concept relevant to HDPS.

## 2. Imports

We begin by loading the Python packages and VITAL modules used throughout the case study.

In [ ]:
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from urllib3.exceptions import InsecureRequestWarning

from vital.module_tidal import process_tidal_data
from vital.module_rotor import RotorData
from vital.module_rotor_simulation import RotorSimulation
from vital.module_vessel import VesselData
from vital.module_constraint_checker import ConstraintChecker
from vital.module_lcoe import LCOEData, LCOECalculator
from vital.module_lcoe_optimizer import LCOEOptimizer

warnings.simplefilter("ignore", InsecureRequestWarning)

plt.style.use("tableau-colorblind10")

## 3. Baseline design assumptions

To keep the comparison clear, we start with a fixed baseline design.

The baseline design includes:

- rotor size,
- rated power,
- hub depth,
- drivetrain settings,
- vessel/platform geometry,
- and cost assumptions.

These values are representative inputs for a screening-level case study and should not be interpreted as final HDPS design parameters.

Later, we will compare this baseline design across multiple sites and then test whether the design can be improved through optimization.

In [ ]:
# Baseline turbine and drivetrain configuration
config = {
    # Turbine geometry and ratings
    "Radius": 3.0,             # Rotor radius (m)
    "Prated": 150000.0,        # Rated electrical power per turbine (W)
    "Trated": 10000.0,         # Rated generator torque (N m)
    "dHub": 5.0,               # Hub depth below free surface (m)
    "number_of_turbines": 4,   # Number of turbines in the system

    # Drivetrain and generator parameters
    "Ng": 5.0,                 # Gear ratio
    "J_d": 50.0,               # Drivetrain inertia
    "B_d": 1.0,                # Drivetrain damping/friction
    "J_r": 300.0,              # Rotor inertia
    "power_model": "simple",   # Built-in generator electrical model
    "Kt": 20.0,                # Generator torque constant
    "Rw": 0.1,                 # Generator winding resistance
}

# Representative vessel / support platform assumptions.
# These dimensions are in meters and are roughly representative of a large
# container ship scale. They are used here to define representative support-body
# area and displaced-volume estimates for a screening-level case study.
length_m = 366.0
beam_m = 51.0
draft_m = 15.5
block_coefficient = 0.65

# Simplified projected frontal area used for drag calculations.
projected_area_m2 = beam_m * draft_m

# Approximate displaced volume using a block-coefficient estimate.
vessel_volume_m3 = length_m * beam_m * draft_m * block_coefficient

# Representative vessel/platform properties used by the pitch constraint.
vessel_properties = {
    "Xm": 10.0,                       # Horizontal force-application distance (m)
    "Zm": 5.0,                        # Vertical force-application distance (m)
    "Kphi": 1.0e12,                   # Representative pitch hydrostatic stiffness (N m/rad)
    "theta": np.deg2rad(45.0),        # Mooring line angle (rad)
    "phi": np.deg2rad(5.0),           # Representative pitch angle (rad)
    "area": projected_area_m2,        # Cross-sectional/projected area (m^2)
    "Cd": 0.9,                        # Drag coefficient
}

print("Baseline turbine configuration:")
print(f"  Rotor radius: {config['Radius']:.2f} m")
print(f"  Rated power per turbine: {config['Prated'] / 1000:.1f} kW")
print(f"  Number of turbines: {config['number_of_turbines']}")
print(f"  Total rated power: {config['Prated'] * config['number_of_turbines'] / 1000:.1f} kW")

print()
print("Representative container-ship-scale support body:")
print(f"  Length: {length_m:.1f} m")
print(f"  Beam: {beam_m:.1f} m")
print(f"  Draft: {draft_m:.1f} m")
print(f"  Block coefficient: {block_coefficient:.2f}")
print(f"  Projected area: {projected_area_m2:.1f} m^2")
print(f"  Approximate displaced volume: {vessel_volume_m3:,.0f} m^3")

## 4. Load rotor performance data

The rotor performance data defines how the turbine behaves as flow conditions change.

The rotor file provides:

- ``TSR``: tip-speed ratio
- ``Ct``: thrust coefficient, which indicates the hydrodynamic load the rotor places on the system
- ``Cq``: torque coefficient, which indicates the torque required or produced by the rotor

VITAL computes the power coefficient internally as:

$$
C_p = TSR \cdot C_q
$$

These curves are used by the simulation to estimate power production and loading.

This HDPS case study also loads optional ``Cpmin`` data for the cavitation constraint.

After loading the rotor data, we add the rotor-specific coefficient functions and operating points to the baseline configuration so ``RotorSimulation`` can use them.

The plot below shows how the rotor coefficients vary with tip-speed ratio over the measured TSR range. This helps identify where the turbine performs best and avoids emphasizing extrapolated behavior outside the supplied rotor data.

In [ ]:
rotor = RotorData(
    filename="../data/Sandia_rotor_data.txt",
    cpmin_filename="../data/Cpmin_data.json",
)

print(f"Optimal Cp: {rotor.CpOpt:.3f}")
print(f"Optimal TSR: {rotor.TSROpt:.3f}")
print(f"Estimated TSRmax: {rotor.TSRmax:.3f}")
print(f"Measured TSR range: {rotor.tsr.min():.3f} to {rotor.tsr.max():.3f}")

# Add rotor-specific coefficient functions and operating points to the config.
config["CpFunc"] = rotor.get_cp
config["CqFunc"] = rotor.get_cq
config["CtFunc"] = rotor.get_ct
config["CpOpt"] = rotor.CpOpt
config["TSROpt"] = rotor.TSROpt
config["TSRmax"] = rotor.TSRmax

# Plot over the measured TSR range to avoid emphasizing extrapolated behavior.
tsr_values = np.linspace(rotor.tsr.min(), rotor.tsr.max(), 200)

cp_values = rotor.get_cp(tsr_values)
ct_values = rotor.get_ct(tsr_values)
cq_values = rotor.get_cq(tsr_values)

plt.figure(figsize=(10, 5))
plt.plot(tsr_values, cp_values, label="Cp")
plt.plot(tsr_values, ct_values, label="Ct")
plt.plot(tsr_values, cq_values, label="Cq")
plt.axvline(rotor.TSROpt, color="k", linestyle="--", linewidth=1, label="Optimal TSR")
plt.xlabel("Tip-speed ratio (TSR)")
plt.ylabel("Coefficient")
plt.title("Rotor performance curves over measured TSR range")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

## 5. Load and inspect the tidal resource

We now choose candidate tidal sites and inspect their tidal resources.

For this public-facing case study, we compare three NOAA current stations:

- ``COI1210``
- ``COI0306``
- ``COI0303``

Each site has different tidal conditions and deployment characteristics, so the best design may differ from site to site. Flow speed, mooring depth, and cable length can all affect feasibility and LCOE.

This case study retrieves tidal data directly from NOAA. For offline or fully reproducible documentation builds, use the local-file workflow from the tidal-data tutorial or configure notebook execution accordingly.

In [ ]:
startdate = "2020-01-01"
range_hrs = 14 * 24

# Use hourly data for documentation/runtime practicality.
# For higher-fidelity studies, reduce this value after confirming runtime.
time_step_s = 3600

sites = ["COI1210", "COI0306", "COI0303"]

## 6. Site comparison: baseline design at each site

A key question in tidal energy planning is whether one site is better than another.

Here we keep the design fixed and compare sites to see how resource quality, depth, and cable length affect performance and feasibility.

This section evaluates the same baseline turbine and support-body assumptions at each site. Site-specific optimization is performed later in the case study.

In [ ]:
site_summary = []

for site in sites:
    print(f"Evaluating baseline design at site {site}...")

    tidal_site = process_tidal_data(
        station=site,
        startdate=startdate,
        range_hrs=range_hrs,
        time_step_s=time_step_s,
        city_data_file="../data/AlaskaCityLatLong.txt",
    )

    config_site = config.copy()
    config_site["dMoor"] = tidal_site.mooring_distance
    config_site["Uinf"] = tidal_site.flow_speeds
    config_site["t"] = tidal_site.times

    rotor_sim = RotorSimulation(config_site)
    rotor_sim.simulate()
    sim_site = rotor_sim.get_results()

    vessel_site = VesselData(
        user_defined=True,
        vessel_properties=vessel_properties,
        simResult=sim_site,
    )

    checker_site = ConstraintChecker(rotor, config_site, vessel_site, sim_site)

    constraint_results = {
        "Power": checker_site.check_power_constraint(),
        "Depth": checker_site.check_depth_constraint(),
        "Cavitation": checker_site.check_cavitation_constraint(),
        "Pitch": checker_site.check_pitch_constraint(),
    }

    # Constraint margins provide more detail than pass/fail flags.
    power_margin_min = np.min(checker_site.power_constraint())
    depth_margin = checker_site.depth_constraint()
    cavitation_margin_min = np.min(checker_site.cavitation_constraint())
    pitch_margin_min = np.min(checker_site.pitch_constraint())

    feasible = all(constraint_results.values())

    if feasible:
        lcoe_data_site = LCOEData(
            tidalData=tidal_site,
            turbineConfig=config_site,
            vesselData=vessel_site,
            simResult=sim_site,
            lifetime=20,
            discount_rate=0.08,
            turbulence_intensity=0.0,
            customer="customer_A",
            application="grid_connection",
        )
        calc_site = LCOECalculator(lcoe_data_site)
        lcoe_site = calc_site.calculate_lcoe()
        annual_energy_site = calc_site.calculate_annual_energy()
    else:
        lcoe_site = np.nan
        annual_energy_site = np.nan

    site_summary.append(
        {
            "Site": site,
            "Station name": tidal_site.station_name,
            "Nearest city": tidal_site.nearest_city,
            "Mooring depth (m)": tidal_site.mooring_distance,
            "Cable length (m)": tidal_site.cable_length,
            "Mean flow speed (m/s)": np.mean(tidal_site.flow_speeds),
            "Max flow speed (m/s)": np.max(tidal_site.flow_speeds),
            "Annual energy (kWh)": annual_energy_site,
            "LCOE ($/kWh)": lcoe_site,
            "Feasible": feasible,
            "Power ok": constraint_results["Power"],
            "Depth ok": constraint_results["Depth"],
            "Cavitation ok": constraint_results["Cavitation"],
            "Pitch ok": constraint_results["Pitch"],
            "Min power margin (W)": power_margin_min,
            "Depth margin (m)": depth_margin,
            "Min cavitation margin (Pa)": cavitation_margin_min,
            "Min pitch margin (N m)": pitch_margin_min,
        }
    )

site_df = pd.DataFrame(site_summary)
site_df

### Interpretation

The sites differ in both resource quality and deployment conditions. Although higher flow can improve energy capture, longer cable lengths and other site-specific factors can increase cost.

The best site is therefore not determined by flow speed alone. Feasibility also depends on the constraint checks and margins, including power limits, rotor submergence, cavitation, and pitch stability.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4))

# Site resource comparison
axs[0].bar(site_df["Site"], site_df["Mean flow speed (m/s)"])
axs[0].set_ylabel("Mean flow speed (m/s)")
axs[0].set_title("Site resource comparison")
axs[0].grid(True, axis="y", alpha=0.3)

# Site economic comparison
bar_colors = site_df["Feasible"].map({True: "tab:blue", False: "tab:gray"})

axs[1].bar(site_df["Site"], site_df["LCOE ($/kWh)"], color=bar_colors)
axs[1].set_ylabel("LCOE ($/kWh)")
axs[1].set_title("Site economic comparison")
axs[1].grid(True, axis="y", alpha=0.3)

# Label infeasible sites if any
for idx, row in site_df.iterrows():
    if not row["Feasible"]:
        axs[1].text(
            idx,
            0,
            "Infeasible",
            ha="center",
            va="bottom",
            rotation=90,
            color="tab:red",
            fontsize=9,
        )

plt.tight_layout()
plt.show()

## 7. Detailed simulation at one site

Next, we focus on one site in more detail.

This lets us examine:

- adjusted inflow speed,
- rotor speed,
- tip-speed ratio,
- electrical power,
- generator torque,
- and thrust force.

These plots help users understand how the turbine responds over time.

The selected site is used as a representative detailed example. It is not necessarily the best site unless supported by the site-comparison and optimization results.

In [ ]:
site = "COI1210"

tidal = process_tidal_data(
    station=site,
    startdate=startdate,
    range_hrs=range_hrs,
    time_step_s=time_step_s,
    city_data_file="../data/AlaskaCityLatLong.txt",
)

print(f"Detailed site: {site}")
print(f"Station name: {tidal.station_name}")
print(f"Nearest city: {tidal.nearest_city}")
print(f"Mooring depth: {tidal.mooring_distance:.2f} m")
print(f"Cable length: {tidal.cable_length:.2f} m")
print(f"Number of time points: {len(tidal.times)}")

config_base = config.copy()
config_base["dMoor"] = tidal.mooring_distance
config_base["Uinf"] = tidal.flow_speeds
config_base["t"] = tidal.times

rotor_sim = RotorSimulation(config_base)
rotor_sim.simulate()
sim_base = rotor_sim.get_results()

vessel_base = VesselData(
    user_defined=True,
    vessel_properties=vessel_properties,
    simResult=sim_base,
)

print("Detailed site simulation complete.")
print(f"Mean electrical power: {np.mean(sim_base['Pelec']) / 1000:.2f} kW")
print(f"Maximum electrical power: {np.max(sim_base['Pelec']) / 1000:.2f} kW")

In [ ]:
time_days = sim_base["t"] / (3600 * 24)

fig, axs = plt.subplots(2, 2, figsize=(12, 8), sharex=True)

# Adjusted and surface inflow speed
axs[0, 0].plot(time_days, sim_base["Uinf_adjusted"], lw=1.8, label="Adjusted inflow")
axs[0, 0].plot(time_days, tidal.flow_speeds, lw=1.2, alpha=0.7, label="Surface/current input")
axs[0, 0].set_ylabel("Speed (m/s)")
axs[0, 0].set_title("Flow Speed")
axs[0, 0].grid(True, alpha=0.3)
axs[0, 0].legend()

# Rotor angular speed
axs[0, 1].plot(time_days, sim_base["wr"], lw=1.8)
axs[0, 1].set_ylabel("Rotor speed (rad/s)")
axs[0, 1].set_title("Rotor Angular Speed")
axs[0, 1].grid(True, alpha=0.3)

# Tip-speed ratio
axs[1, 0].plot(time_days, sim_base["TSR"], lw=1.8, label="TSR")
axs[1, 0].axhline(config_base["TSROpt"], color="k", linestyle="--", label="Optimal TSR")
axs[1, 0].axhline(config_base["TSRmax"], color="r", linestyle=":", label="Estimated TSRmax")
axs[1, 0].set_xlabel("Time (days)")
axs[1, 0].set_ylabel("TSR (-)")
axs[1, 0].set_title("Tip-Speed Ratio")
axs[1, 0].grid(True, alpha=0.3)
axs[1, 0].legend()

# Electrical power
axs[1, 1].plot(time_days, sim_base["Pelec"] / 1000.0, lw=1.8, label="Electrical power")
axs[1, 1].axhline(
    config_base["Prated"] / 1000.0,
    color="r",
    linestyle="--",
    label="Rated power",
)
axs[1, 1].set_xlabel("Time (days)")
axs[1, 1].set_ylabel("Power (kW)")
axs[1, 1].set_title("Electrical Power")
axs[1, 1].grid(True, alpha=0.3)
axs[1, 1].legend()

plt.tight_layout()
plt.show()

In [ ]:
time_days = sim_base["t"] / (3600 * 24)

fig, axs = plt.subplots(2, 1, figsize=(10, 7), sharex=True)

# Generator torque
axs[0].plot(time_days, sim_base["Tg"], lw=1.8, label="Generator torque")
axs[0].axhline(
    config_base["Trated"],
    color="r",
    linestyle="--",
    label="Rated torque",
)
axs[0].set_ylabel("Torque (N m)")
axs[0].set_title("Generator Torque")
axs[0].grid(True, alpha=0.3)
axs[0].legend()

# Rotor thrust force
axs[1].plot(time_days, sim_base["Ft"] / 1000.0, lw=1.8, label="Rotor thrust")
axs[1].set_xlabel("Time (days)")
axs[1].set_ylabel("Thrust (kN)")
axs[1].set_title("Rotor Thrust Force")
axs[1].grid(True, alpha=0.3)
axs[1].legend()

plt.tight_layout()
plt.show()

## 8. Constraint checking

A design is not useful unless it is physically feasible.

The software checks several key constraints:

- power limit: electrical power should remain nonnegative and below rated power,
- depth/submergence: the rotor should remain below the water surface,
- cavitation margin: estimated blade pressure should remain above vapor pressure,
- pitch stability: simplified destabilizing moments should remain below the restoring pitch moment.

These checks help identify whether a design is operable at the selected site.

The pass/fail results are useful, but the constraint margins are also important because they show how close the design is to violating each constraint.

In [ ]:
checker_base = ConstraintChecker(rotor, config_base, vessel_base, sim_base)

constraint_results = {
    "Power Constraint": checker_base.check_power_constraint(),
    "Depth Constraint": checker_base.check_depth_constraint(),
    "Cavitation Constraint": checker_base.check_cavitation_constraint(),
    "Pitch Constraint": checker_base.check_pitch_constraint(),
}

print("Constraint results:")
for name, passed in constraint_results.items():
    print(f"  {name}: {'PASS' if passed else 'FAIL'}")

# Constraint margins
power_margin = checker_base.power_constraint()
depth_margin = checker_base.depth_constraint()
cavitation_margin = checker_base.cavitation_constraint()
pitch_margin = checker_base.pitch_constraint()

print()
print("Constraint margins:")
print(f"  Minimum power margin, Prated - Pelec: {np.min(power_margin):.3f} W")
print(f"  Depth margin, dHub - Radius: {depth_margin:.3f} m")
print(f"  Minimum cavitation margin: {np.min(cavitation_margin):.3f} Pa")
print(f"  Minimum pitch margin: {np.min(pitch_margin):.3f} N m")

## 9. LCOE calculation

The next step is to translate the simulation results into an economic metric.

LCOE combines:

- upfront capital cost,
- annual operating cost,
- and annual energy production.

This case study uses the HDPS-style grid-connected cost configuration:

- ``customer="customer_A"``
- ``application="grid_connection"``

The result is reported in USD/kWh and should be interpreted as a screening-level estimate that depends on the selected cost model and input assumptions.

In [ ]:
lcoe_data_base = LCOEData(
    tidalData=tidal,
    turbineConfig=config_base,
    vesselData=vessel_base,
    simResult=sim_base,
    lifetime=20,
    discount_rate=0.08,
    turbulence_intensity=0.0,
    customer="customer_A",
    application="grid_connection",
)

calc_base = LCOECalculator(lcoe_data_base)

capex_summary = calc_base.get_capex_summary()
component_capex = capex_summary["component_capex"]
adjusted_capex = capex_summary["adjusted_capex"]
adjustment_factor = capex_summary["adjustment_factor"]

total_opex = calc_base.calculate_total_opex(adjusted_capex)
annual_energy = calc_base.calculate_annual_energy()
baseline_lcoe = calc_base.calculate_lcoe()

print(f"Total component CAPEX: ${component_capex:,.2f}")
print(f"Adjustment factor: {adjustment_factor:.2f}")
print(f"Adjusted CAPEX: ${adjusted_capex:,.2f}")
print(f"Total OPEX: ${total_opex:,.2f} per year")
print(f"Annual Energy: {annual_energy:,.2f} kWh/year")
print(f"LCOE: ${baseline_lcoe:.4f}/kWh")

### CAPEX summary and adjusted cost

The cost model reports both:

- the component CAPEX, which is the sum of the itemized project costs,
- the adjusted CAPEX used in the LCOE calculation.

For this HDPS-style case study, ``customer_A`` applies a CAPEX adjustment factor of ``1.05``.

### OPEX and energy summary

After CAPEX is calculated, the notebook reports:

- OPEX, or annual operating cost,
- annual energy generation, which is used to estimate LCOE.

The current active OPEX model assumes annual OPEX is 4% of adjusted CAPEX.

In [ ]:
capex_summary = calc_base.get_capex_summary()

print("CAPEX Summary:")
print(f"  Total component CAPEX: ${capex_summary['component_capex']:,.2f}")
print(f"  Adjustment factor: {capex_summary['adjustment_factor']:.2f}")
print(f"  Final CAPEX used in LCOE: ${capex_summary['adjusted_capex']:,.2f}")

print("\nCAPEX Items:")
for cost_name, cost_value in capex_summary["capex_items"].items():
    print(f"  {cost_name}: ${cost_value:,.2f}")

total_opex = calc_base.calculate_total_opex(capex_summary["adjusted_capex"])
annual_energy = calc_base.calculate_annual_energy()

print("\nOPEX Summary:")
print(f"  Annual OPEX: ${total_opex:,.2f}/year")

print("\nEnergy Summary:")
print(f"  Annual Energy Generation: {annual_energy:,.2f} kWh/year")

The CAPEX breakdown shows the component CAPEX items used in the model.  
For HDPS (``customer_A``), the final CAPEX includes a 5% customer-specific adjustment factor applied to the total component CAPEX. This factor can be interpreted as a simplified allowance for project-specific adjustment, overhead, or contingency.

## 10. Optimization study

Now we ask a practical design question:

**Can the baseline design be improved by changing a few key design variables?**

Here we vary:

- rotor radius,
- rated power,
- hub depth,
- and number of turbines.

The optimizer performs an explicit grid search over the selected design-variable bounds. For each candidate design, it runs the rotor simulation, checks constraints, and calculates LCOE for feasible designs.

The goal is to find a lower-LCOE design while still satisfying the physical constraints.

For simplicity, this optimization varies a small set of design variables while holding drivetrain and generator parameters fixed. In a detailed design study, values such as rated torque, inertia, gear ratio, generator constants, and structural properties may need to scale with rotor size and rated power.

The optimization results reflect the best designs found within the selected search bounds.

In [ ]:
optimization_bounds = {
    "Radius": (2.0, 4.0, 1.0),              # 2.0, 3.0, 4.0 m
    "Prated": (100000.0, 200000.0, 50000.0), # 100, 150, 200 kW
    "dHub": (3.0, 7.0, 2.0),                # 3.0, 5.0, 7.0 m
    "number_of_turbines": (2, 6, 2),        # 2, 4, 6
}

num_grid_points = 1
for low, high, step in optimization_bounds.values():
    values = np.arange(low, high + 0.5 * step, step)
    num_grid_points *= len(values)

print(f"Number of design points to evaluate: {num_grid_points}")

In [ ]:
# Use the detailed baseline site from above.
optimizer = LCOEOptimizer(
    tidal=tidal,
    rotor=rotor,
    base_config=config_base,
    user_vessel_properties=vessel_properties,
)

opt_result_base = optimizer.optimize(
    variable_bounds=optimization_bounds,
    customer="customer_A",
    application="grid_connection",
    lifetime=20,
    discount_rate=0.08,
    turbulence_intensity=0.0,
)

print("Optimal parameters for the detailed site:")
for name, value in opt_result_base["optimal_params"].items():
    print(f"  {name}: {value}")

print(f"\nOptimal LCOE: ${opt_result_base['optimal_lcoe']:.4f}/kWh")
print(f"Total design points: {len(opt_result_base['results_table'])}")
print(f"Feasible design points: {len(opt_result_base['feasible_table'])}")

In [ ]:
optimized_lcoe = opt_result_base["optimal_lcoe"]

print(f"Baseline LCOE:  ${baseline_lcoe:.4f}/kWh")
print(f"Optimized LCOE: ${optimized_lcoe:.4f}/kWh")

if np.isfinite(baseline_lcoe) and np.isfinite(optimized_lcoe):
    improvement = 100.0 * (baseline_lcoe - optimized_lcoe) / baseline_lcoe
    print(f"Relative improvement: {improvement:.2f}%")

    if improvement > 0:
        print("The optimized design improves on the baseline within the selected search bounds.")
    elif np.isclose(improvement, 0.0):
        print("The optimized design has approximately the same LCOE as the baseline.")
    else:
        print("The optimized design is more expensive than the baseline. The search grid did not find an improvement.")

print("\nOptimal parameters:")
for name, value in opt_result_base["optimal_params"].items():
    print(f"  {name}: {value}")

### Interpretation

For this search grid, the optimizer found a lower-LCOE configuration than the baseline. The selected design keeps the same rotor radius, rated power, and hub depth as the baseline, but increases the number of turbines within the selected bounds.

This suggests that, under the current cost model, increasing turbine count can reduce LCOE when the added energy production grows faster than the modeled project costs.

This result should be interpreted in the context of the selected search bounds, cost model, and constraint checks. If turbine, installation, infrastructure, or vessel/platform costs change, the preferred design may also change.

In [ ]:
optimizer.plot_results(opt_result_base)

## 11. Optimization across all three sites

Finally, we repeat the optimization process for all three candidate sites.

This lets us compare how the best design changes with site conditions.

For each site, the optimizer uses that site's tidal resource, mooring depth, cable length, and time series. This comparison highlights that tidal resource quality, deployment depth, and cable length all influence whether a site is attractive for a grid-connected project.

In [ ]:
import contextlib
import io

site_opt_summary = []

for site in sites:
    print(f"\n=== Optimizing site: {site} ===")

    tidal_site = process_tidal_data(
        station=site,
        startdate=startdate,
        range_hrs=range_hrs,
        time_step_s=time_step_s,
        city_data_file="../data/AlaskaCityLatLong.txt",
    )

    config_site = config.copy()
    config_site["dMoor"] = tidal_site.mooring_distance
    config_site["Uinf"] = tidal_site.flow_speeds
    config_site["t"] = tidal_site.times

    # Important: create a site-specific optimizer.
    # This ensures each site uses its own tidal data, mooring depth, cable length, and time series.
    optimizer_site = LCOEOptimizer(
        tidal=tidal_site,
        rotor=rotor,
        base_config=config_site,
        user_vessel_properties=vessel_properties,
    )

    try:
        # Suppress detailed per-design-point output for a cleaner case-study notebook.
        with contextlib.redirect_stdout(io.StringIO()):
            opt_result = optimizer_site.optimize(
                variable_bounds=optimization_bounds,
                customer="customer_A",
                application="grid_connection",
                lifetime=20,
                discount_rate=0.08,
                turbulence_intensity=0.0,
            )

        print(f"  Optimal LCOE: ${opt_result['optimal_lcoe']:.4f}/kWh")
        print(f"  Feasible designs: {len(opt_result['feasible_table'])} of {len(opt_result['results_table'])}")

        site_opt_summary.append(
            {
                "Site": site,
                "Station name": tidal_site.station_name,
                "Nearest city": tidal_site.nearest_city,
                "Mooring depth (m)": tidal_site.mooring_distance,
                "Cable length (m)": tidal_site.cable_length,
                "Mean flow speed (m/s)": np.mean(tidal_site.flow_speeds),
                "Max flow speed (m/s)": np.max(tidal_site.flow_speeds),
                "Optimal LCOE ($/kWh)": opt_result["optimal_lcoe"],
                "Optimal Radius (m)": opt_result["optimal_params"]["Radius"],
                "Optimal Prated (W)": opt_result["optimal_params"]["Prated"],
                "Optimal dHub (m)": opt_result["optimal_params"]["dHub"],
                "Optimal number of turbines": int(opt_result["optimal_params"]["number_of_turbines"]),
                "Feasible designs": len(opt_result["feasible_table"]),
                "Total designs": len(opt_result["results_table"]),
            }
        )

    except Exception as e:
        print(f"  Optimization failed for site {site}: {e}")

        site_opt_summary.append(
            {
                "Site": site,
                "Station name": tidal_site.station_name,
                "Nearest city": tidal_site.nearest_city,
                "Mooring depth (m)": tidal_site.mooring_distance,
                "Cable length (m)": tidal_site.cable_length,
                "Mean flow speed (m/s)": np.mean(tidal_site.flow_speeds),
                "Max flow speed (m/s)": np.max(tidal_site.flow_speeds),
                "Optimal LCOE ($/kWh)": np.nan,
                "Optimal Radius (m)": np.nan,
                "Optimal Prated (W)": np.nan,
                "Optimal dHub (m)": np.nan,
                "Optimal number of turbines": np.nan,
                "Feasible designs": 0,
                "Total designs": np.nan,
            }
        )

site_opt_df = pd.DataFrame(site_opt_summary)
site_opt_df

### Interpretation of optimized site comparison

In this run, the optimizer selected the same design across the three sites within the selected search bounds:

- rotor radius: 3.0 m
- rated power: 150 kW per turbine
- hub depth: 5.0 m
- number of turbines: 6

However, the optimized LCOE differs substantially by site. In this run, ``COI0303`` has the lowest optimized LCOE, even though it does not have the highest mean flow speed. This is likely because ``COI0303`` has a much shorter estimated cable length than the other sites.

This result highlights that site selection is not determined by tidal resource alone. Cable length, mooring depth, feasibility constraints, and cost-model assumptions can strongly influence the final LCOE.

These results should be interpreted within the selected search bounds and cost model. A different cable-routing assumption, cost model, turbine design space, or local tidal dataset could change the preferred site or design.

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(15, 4))

# Optimized LCOE
axs[0].bar(site_opt_df["Site"], site_opt_df["Optimal LCOE ($/kWh)"])
axs[0].set_ylabel("Optimal LCOE ($/kWh)")
axs[0].set_title("Optimized LCOE by site")
axs[0].grid(True, axis="y", alpha=0.3)

# Estimated cable length
axs[1].bar(site_opt_df["Site"], site_opt_df["Cable length (m)"] / 1000)
axs[1].set_ylabel("Cable length (km)")
axs[1].set_title("Estimated cable length by site")
axs[1].grid(True, axis="y", alpha=0.3)

# Mean flow speed
axs[2].bar(site_opt_df["Site"], site_opt_df["Mean flow speed (m/s)"])
axs[2].set_ylabel("Mean flow speed (m/s)")
axs[2].set_title("Mean flow speed by site")
axs[2].grid(True, axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

## 12. Interpretation and takeaway

This case study shows how VITAL can support a full screening-level workflow for a grid-connected tidal energy system:

- compare candidate sites,
- simulate turbine performance,
- check operational constraints,
- estimate project economics,
- and search for improved designs.

For this HDPS-style grid-connected example, the optimized LCOE varied strongly by site. The lowest-LCOE site in this run was not simply the site with the highest mean flow speed; estimated cable length and cost assumptions were also important.

This demonstrates why technical feasibility and economic performance should be evaluated together. A site with strong tidal flow may still be less attractive if infrastructure costs, cable length, or constraint margins are unfavorable.

## Key takeaways

- Site selection matters as much as turbine design.
- The strongest tidal resource is not always the lowest-LCOE option.
- Cable length and infrastructure-related costs can strongly influence grid-connected LCOE.
- Physical constraints are essential to feasibility.
- Optimization results depend on the selected cost model, design-variable bounds, and assumptions.
- VITAL can support both design analysis and site comparison for early-stage screening.